# Before your start:

    Read the README.md file
    Comment as much as you can and use the resources (README.md file)
    Happy learning!

In this exercise, we  will generate random numbers from the continuous disributions we learned in the lesson. There are two ways to generate random numbers:

1. Using the numpy library 
1. using the Scipy library 

Use either or both of the lbraries in this exercise.

In [5]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import math
import seaborn as sns
from scipy import stats
from scipy.stats import norm ,uniform
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## Uniform Distribution

To generate uniform random numbers between any two given values using scipy, we can either use the following code or the code that we have
discussed in class:

In [ ]:
# your code here



**Your task:**

1. Based on the code above, write a function that generates uniformly distributed random numbers. There are several requirements for your function:
    * It should accept 3 parameters: 
        * `bottom` - the lower boundary of the generated numbers
        * `ceiling` - the upper boundary of the generated numbers
        * `count` - how many numbers to generate
    * It should return an array of uniformly distributed random numbers

1. Call your function with 2 sets of params below:
    * bottom=10, ceiling=15, count=100
    * bottom=10, ceiling=60, count=1,000

1. Plot the uniform distributions generated above using histograms, where x axis is the value and y axis is the count. Let the histogram's number of bins be 10.

Your output should look like below:

![uniform distribution](ud.png)

In [3]:
# your code here
def generate_uniform(bottom: float, ceiling: float, count: int, seed: int = None) -> dict:
    # Validate inputs with assert
    assert isinstance(count, int) and count > 0, "count must be a positive integer"
    assert bottom < ceiling, "bottom must be strictly less than ceiling"

    # np.random.default_rng is the modern, thread-safe generator
    # seed makes results reproducible across runs
    rng = np.random.default_rng(seed)

    # Generate the uniform numbers
    numbers = rng.uniform(low=bottom, high=ceiling, size=count)

    # Return both data and stats in a structured dict
    return {
        "data"  : numbers,
        "mean"  : round(numbers.mean(), 4),
        "std"   : round(numbers.std(), 4),
        "min"   : round(numbers.min(), 4),
        "max"   : round(numbers.max(), 4),
        "count" : len(numbers)
    }

In [4]:
# your code here
import pandas as pd

configs = [
    {"bottom": 10, "ceiling": 15, "count": 100,  "seed": 42, "label": "Set 1: range[10–15], n=100"},
    {"bottom": 10, "ceiling": 60, "count": 1000, "seed": 42, "label": "Set 2: range[10–60], n=1000"},
]

summary_rows = []
expert_results = {}

for cfg in configs:
    label  = cfg.pop("label")              # extract label key before passing to function
    result = generate_uniform(**cfg) # remaining keys match function parameters
    expert_results[label] = result["data"]
    summary_rows.append({
        "Label"  : label,
        "Count"  : result["count"],
        "Mean"   : result["mean"],
        "Std Dev": result["std"],
        "Min"    : result["min"],
        "Max"    : result["max"],
    })

# Build a clean summary DataFrame — useful for reporting
df_summary = pd.DataFrame(summary_rows)
print(df_summary.to_string(index=False))

                      Label  Count    Mean  Std Dev     Min     Max
 Set 1: range[10–15], n=100    100 12.4336   1.3612 10.0368 14.8781
Set 2: range[10–60], n=1000   1000 34.8589  14.5720 10.0617 59.9552


In [ ]:
# your code here
from scipy.stats import gaussian_kde

#  Generate datasets (must be in the same cell or run before this) 
rng  = np.random.default_rng(42)          # seed=42 for reproducibility
set1 = rng.uniform(low=10, high=15, size=100)    # narrow range, small sample
set2 = rng.uniform(low=10, high=60, size=1000)   # wide range, large sample

#  Create subplot figure with 2 panels 
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Set 1: range[10–15], n=100",
                                    "Set 2: range[10–60], n=1000"])

datasets = [(set1, 10, 15, 1), (set2, 10, 60, 2)]

for data, bot, ceil, col in datasets:
    # histnorm="probability density" scales y so the total area = 1
    # this makes the histogram directly comparable to a PDF curve
    fig.add_trace(go.Histogram(
        x=data, nbinsx=10, histnorm="probability density", opacity=0.6
    ), row=1, col=col)

    # KDE = Kernel Density Estimate — a smooth curve that approximates
    # the true shape of the distribution from the data
    kde   = gaussian_kde(data)
    x_kde = np.linspace(bot, ceil, 300)   # 300 evenly-spaced x points
    fig.add_trace(go.Scatter(
        x=x_kde, y=kde(x_kde), mode="lines", name=f"KDE Set {col}"
    ), row=1, col=col)

    # Theoretical uniform PDF = flat line at height 1/(ceil - bot)
    # For Set 1: 1/(15-10) = 0.20 | For Set 2: 1/(60-10) = 0.02
    y_theory = 1 / (ceil - bot)
    fig.add_trace(go.Scatter(
        x=[bot, ceil], y=[y_theory, y_theory],
        mode="lines", line=dict(dash="dash"), name=f"Theoretical PDF {col}"
    ), row=1, col=col)

##fig.write_image("uniform_expert.png")
fig.show()

How are the two distributions different?

In [11]:
# your answer below
from scipy.stats import gaussian_kde

fig = make_subplots(rows=1, cols=2)

for data, bot, ceil, col, name in [(set1,10,15,1,"Set 1"),(set2,10,60,2,"Set 2")]:

    # Fit KDE to data → smooth density curve
    kde   = gaussian_kde(data)
    x_kde = np.linspace(bot - 1, ceil + 1, 400)  # extend slightly beyond range

    # fill="tozeroy" fills the area under the curve → easy to see spread visually
    fig.add_trace(go.Scatter(
        x=x_kde, y=kde(x_kde),
        fill="tozeroy", mode="lines", name=name
    ), row=1, col=col)

    # add_vline draws a vertical dashed line at the mean value
    fig.add_vline(x=data.mean(), line_dash="dash",
                  annotation_text=f"mean={data.mean():.1f}",
                  row=1, col=col)
    fig.show()

## Normal Distribution

1. In the same way in the Uniform Distribution challenge, write a function that generates normally distributed random numbers.
1. Generate 1,000 normally distributed numbers with the average of 10 and standard deviation of 1
1. Generate 1,000 normally distributed numbers with the average of 10 and standard deviation of 50
2. Plot the distributions of the data generated.

Expected output:

![normal distribution](nd.png)

In [12]:
from scipy.stats import norm 

In [13]:
# your code here
def generate_normal(mean: float, std: float, count: int, seed: int = None) -> dict:
    # Validate with assertions — fail early and clearly
    assert std > 0,   f"std must be positive, got {std}"
    assert count > 0, f"count must be positive, got {count}"

    # np.random.default_rng is the modern, recommended generator
    # seed makes every run produce the same numbers — critical for debugging
    rng     = np.random.default_rng(seed)
    numbers = rng.normal(loc=mean, scale=std, size=count)

    return {
        "data"    : numbers,
        "mean"    : round(numbers.mean(), 4),
        "std"     : round(numbers.std(),  4),
        "min"     : round(numbers.min(),  4),
        "max"     : round(numbers.max(),  4),
        "count"   : len(numbers),
        # Skewness measures asymmetry: 0 = perfect bell curve
        # Formula: mean of ((x - mean) / std)^3
        "skewness": round(float(np.mean(((numbers - numbers.mean()) / numbers.std()) ** 3)), 4)
    }

In [14]:
# your code here

set1 = generate_normal(mean=10, std=1, count=1000, seed=42)

# np.percentile returns the value below which X% of data falls
# P50 = median, should be very close to mean for a normal distribution
percentiles = [1, 5, 10, 25, 50, 75, 90, 95, 99]
perc_values = np.percentile(set1["data"], percentiles)

for p, v in zip(percentiles, perc_values):
    print(f"P{p:<3} : {v:.4f}")

P1   : 7.6658
P5   : 8.3144
P10  : 8.7169
P25  : 9.3037
P50  : 10.0062
P75  : 10.5899
P90  : 11.2540
P95  : 11.6162
P99  : 12.1427


In [17]:
# your code here
set1 = generate_normal(mean=10, std=1,  count=1000, seed=42)
set2 = generate_normal(mean=10, std=50, count=1000, seed=42)

# Notice: skewness is IDENTICAL (-0.0437) for both
# because same seed = same underlying random pattern,
# just scaled by std — confirms std only stretches, doesn't distort shape
percentiles = [1, 5, 25, 50, 75, 95, 99]
for p in percentiles:
    v1 = np.percentile(set1["data"], p)
    v2 = np.percentile(set2["data"], p)
    print(f"P{p:<3}: std=1 → {v1:.2f}  | std=50 → {v2:.2f}")

P1  : std=1 → 7.67  | std=50 → -106.71
P5  : std=1 → 8.31  | std=50 → -74.28
P25 : std=1 → 9.30  | std=50 → -24.82
P50 : std=1 → 10.01  | std=50 → 10.31
P75 : std=1 → 10.59  | std=50 → 39.49
P95 : std=1 → 11.62  | std=50 → 90.81
P99 : std=1 → 12.14  | std=50 → 117.13


In [19]:
# your code here
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import norm, gaussian_kde

# ── STEP 1: Redefine the expert generator ────────────────────────────────────
def generate_normal(mean: float, std: float, count: int, seed: int = None) -> dict:
    assert std > 0,   f"std must be positive, got {std}"
    assert count > 0, f"count must be positive, got {count}"
    rng     = np.random.default_rng(seed)
    numbers = rng.normal(loc=mean, scale=std, size=count)
    return {
        "data"    : numbers,           # <-- the actual array lives here
        "mean"    : round(numbers.mean(), 4),
        "std"     : round(numbers.std(),  4),
        "min"     : round(numbers.min(),  4),
        "max"     : round(numbers.max(),  4),
        "count"   : len(numbers),
        "skewness": round(float(np.mean(((numbers - numbers.mean()) / numbers.std()) ** 3)), 4)
    }

# ── STEP 2: Generate both sets — returns DICTS, not arrays ───────────────────
set1_dict = generate_normal(mean=10, std=1,  count=1000, seed=42)
set2_dict = generate_normal(mean=10, std=50, count=1000, seed=42)

# ── STEP 3: Extract the actual arrays using ["data"] ─────────────────────────
# THIS is the fix — you must pull the array out of the dict before plotting
set1 = set1_dict["data"]   # numpy array — safe to pass to Plotly
set2 = set2_dict["data"]   # numpy array — safe to pass to Plotly

# ── STEP 4: Plot ──────────────────────────────────────────────────────────────
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Set 1: mean=10, std=1",
                                    "Set 2: mean=10, std=50"])

for data, mean, std, col in [(set1, 10, 1, 1), (set2, 10, 50, 2)]:

    # Density histogram — now 'data' is a clean numpy array, not a dict
    fig.add_trace(go.Histogram(
        x=data,                             # ✅ array, not dict
        nbinsx=30,
        histnorm="probability density",
        opacity=0.55,
        name=f"Histogram std={std}"
    ), row=1, col=col)

    # KDE smooth curve
    kde   = gaussian_kde(data)
    x_kde = np.linspace(data.min(), data.max(), 400)
    fig.add_trace(go.Scatter(
        x=x_kde, y=kde(x_kde),
        mode="lines", name=f"KDE std={std}"
    ), row=1, col=col)

    # Theoretical PDF — perfect bell curve
    x_pdf = np.linspace(mean - 4*std, mean + 4*std, 400)
    y_pdf = norm.pdf(x_pdf, loc=mean, scale=std)
    fig.add_trace(go.Scatter(
        x=x_pdf, y=y_pdf,
        mode="lines", line=dict(dash="dash"),
        name=f"Theoretical PDF std={std}"
    ), row=1, col=col)

for col in [1, 2]:
    fig.update_xaxes(title_text="Value",   row=1, col=col)
    fig.update_yaxes(title_text="Density", row=1, col=col)

fig.show()

How are the two distributions different?

In [20]:
fig = go.Figure()

for data, mean, std, label in [(set1,10,1,"std=1"),(set2,10,50,"std=50")]:
    kde   = gaussian_kde(data)
    x_kde = np.linspace(data.min(), data.max(), 600)

    # fill="tozeroy" fills under the curve — makes spread differences obvious
    fig.add_trace(go.Scatter(
        x=x_kde, y=kde(x_kde),
        fill="tozeroy", mode="lines", name=label, opacity=0.6
    ))

# Add reference lines at mean and ±1σ, ±2σ for std=1
# This visually proves how tiny std=1's range is compared to std=50
for x_val, label in [(10,"mean"),(9,"−1σ"),(11,"+1σ"),(8,"−2σ"),(12,"+2σ")]:
    fig.add_vline(x=x_val, line_dash="dot",
                  annotation_text=label, annotation_position="top")

fig.update_xaxes(title_text="Value", range=[-200, 200])
fig.update_yaxes(title_text="Density")
fig.show()

## Normal Distribution of Real Data

In this challenge we are going to take a look the real data. We will use vehicles.csv file for this exercise

In [27]:
# your code here
import pandas as pd

# pd.read_csv reads any CSV file into a DataFrame
# A DataFrame is like an Excel table — rows are records, columns are variables
df = pd.read_csv("vehicles.csv")
target_cols = ["Fuel Barrels/Year", "CO2 Emission Grams/Mile", "Combined MPG"]

# .describe() gives the full statistical summary in one line
print(df[target_cols].describe().round(4))

# .skew() measures asymmetry — critical before assuming normal distribution
# > 0 = tail on the right (right-skewed)
# < 0 = tail on the left (left-skewed)
# ~0  = symmetric (close to normal)
for col in target_cols:
    print(f"{col}: skewness = {df[col].skew():.4f}")


       Fuel Barrels/Year  CO2 Emission Grams/Mile  Combined MPG
count         35952.0000               35952.0000    35952.0000
mean             17.6091                 475.3163       19.9293
std               4.4673                 119.0608        5.1124
min               0.0600                  37.0000        7.0000
25%              14.6994                 395.0000       16.0000
50%              17.3479                 467.7368       19.0000
75%              20.6006                 555.4375       23.0000
max              47.0871                1269.5714       56.0000
Fuel Barrels/Year: skewness = 0.6383
CO2 Emission Grams/Mile: skewness = 0.7417
Combined MPG: skewness = 1.0678


First import vehicles.csv.
Then plot the histograms for the following variables:

1. Fuel Barrels/Year

In [23]:
# your code here
from scipy.stats import norm, gaussian_kde

fig = go.Figure()

# Layer 1: density histogram
fig.add_trace(go.Histogram(
    x=data, nbinsx=30,
    histnorm="probability density",
    opacity=0.55, name="Histogram"
))

# Layer 2: KDE — smooth curve that follows the ACTUAL shape of your data
kde   = gaussian_kde(data)
x_kde = np.linspace(data.min(), data.max(), 400)
fig.add_trace(go.Scatter(
    x=x_kde, y=kde(x_kde), mode="lines", name="KDE (actual shape)"
))

# Layer 3: norm.fit() finds the best-fit mean and std
# if data were perfectly normal, the dashed line would match the KDE exactly
# the GAP between them is visual proof of non-normality
fitted_mean, fitted_std = norm.fit(data)
x_pdf = np.linspace(data.min(), data.max(), 400)
fig.add_trace(go.Scatter(
    x=x_pdf, y=norm.pdf(x_pdf, loc=fitted_mean, scale=fitted_std),
    mode="lines", line=dict(dash="dash"),
    name=f"Normal fit (μ={fitted_mean:.1f}, σ={fitted_std:.1f})"
))

fig.show()

2. CO2 Emission Grams/Mile 

In [24]:
# your code here
from scipy.stats import norm, gaussian_kde

fig = go.Figure()

# Layer 1: Density histogram — each bar's height is probability density
fig.add_trace(go.Histogram(
    x=data, nbinsx=30,
    histnorm="probability density",
    opacity=0.55, name="Histogram"
))

# Layer 2: KDE — non-parametric smooth curve fitted to real data
# no assumption about distribution shape — just follows the data
kde   = gaussian_kde(data)
x_kde = np.linspace(data.min(), data.max(), 400)
fig.add_trace(go.Scatter(
    x=x_kde, y=kde(x_kde), mode="lines", name="KDE (actual shape)"
))

# Layer 3: norm.fit() finds best μ and σ for a hypothetical normal fit
# the RIGHT TAIL on the KDE is longer than on this dashed line
# → that gap is exactly what "right-skewed, not normal" looks like visually
fitted_mean, fitted_std = norm.fit(data)
x_pdf = np.linspace(data.min(), data.max(), 400)
fig.add_trace(go.Scatter(
    x=x_pdf, y=norm.pdf(x_pdf, loc=fitted_mean, scale=fitted_std),
    mode="lines", line=dict(dash="dash"),
    name=f"Normal fit (μ={fitted_mean:.1f}, σ={fitted_std:.1f})"
))

fig.update_xaxes(title_text="CO2 g/Mile")
fig.update_yaxes(title_text="Density")
fig.show()

3. Combined MPG

In [25]:
# your code here
from scipy.stats import norm, gaussian_kde

fig = go.Figure()

# Layer 1: density histogram
fig.add_trace(go.Histogram(
    x=data, nbinsx=30,
    histnorm="probability density",
    opacity=0.55, name="Histogram"
))

# Layer 2: KDE — smooth empirical curve, no distribution assumption
kde   = gaussian_kde(data)
x_kde = np.linspace(data.min(), data.max(), 400)
fig.add_trace(go.Scatter(
    x=x_kde, y=kde(x_kde), mode="lines", name="KDE (actual shape)"
))

# Layer 3: norm.fit() estimates the best μ and σ if data WERE normal
# Combined MPG has skewness=1.07 — the highest of the 3 columns
# so the right tail separation between KDE and dashed PDF is most visible here
fitted_mean, fitted_std = norm.fit(data)
x_pdf = np.linspace(data.min(), data.max(), 400)
fig.add_trace(go.Scatter(
    x=x_pdf, y=norm.pdf(x_pdf, loc=fitted_mean, scale=fitted_std),
    mode="lines", line=dict(dash="dash"),
    name=f"Normal fit (μ={fitted_mean:.1f}, σ={fitted_std:.1f})"
))

fig.update_xaxes(title_text="Combined MPG")
fig.update_yaxes(title_text="Density")
fig.show()

In [ ]:
# your code here

In [ ]:
# your code here

Which one(s) of the variables are nearly normally distributed? How do you know?

In [28]:
# your code here
from scipy.stats import shapiro

# Shapiro-Wilk tests the null hypothesis: "data IS normally distributed"
# p < 0.05 = reject normality
for col in target_cols:
    data   = df[col].dropna().values
    # Sample 1000 — Shapiro-Wilk is reliable for n <= 5000
    sample = np.random.choice(data, 1000, replace=False)
    _, p   = shapiro(sample)
    print(f"{col}: p = {p:.2e} → {'NORMAL' if p > 0.05 else 'NOT NORMAL'}")

# Q-Q Plot: plot data quantiles vs theoretical normal quantiles
# If data is normal → all points fall on a straight 45° diagonal line
# Curved tails → non-normal
from scipy.stats import probplot
(th_q, ord_data), _ = probplot(df["Combined MPG"].dropna().values, dist="norm")
# Any S-curve or hockey-stick curve in the QQ plot confirms non-normality

Fuel Barrels/Year: p = 3.59e-13 → NOT NORMAL
CO2 Emission Grams/Mile: p = 1.18e-13 → NOT NORMAL
Combined MPG: p = 6.02e-20 → NOT NORMAL


None of them are normally ditributed. 

## Exponential Distribution

1. Using `numpy.random.exponential`, create a function that returns a list of numbers exponentially distributed with the mean of 10. 

1. Use the function to generate two number sequences with the size of 10 and 100.

1. Plot the distributions as histograms with the nubmer of bins as 100.

Your output should look like below:

![exponential distribution](ed.png)

In [30]:
# your code here
def generate_exponential(mean: float, count: int, seed: int = None) -> dict:
    """
    Generate exponentially distributed random numbers.

    Parameters:
        mean  (float): Mean = 1/lambda (average time between events)
        count (int)  : Number of values to generate
        seed  (int)  : Optional seed for reproducibility

    Returns:
        dict: Generated array + key statistics
    """
    # Assert replaces if/raise for cleaner one-liners
    assert mean  > 0, f"mean must be positive, got {mean}"
    assert count > 0, f"count must be positive, got {count}"

    # Modern thread-safe generator with optional seed
    rng     = np.random.default_rng(seed)
    numbers = rng.exponential(scale=mean, size=count)

    return {
        "data"    : numbers,
        "mean"    : round(numbers.mean(), 4),
        "std"     : round(numbers.std(),  4),   # ≈ mean (always, for exp dist)
        "min"     : round(numbers.min(),  4),   # always ≥ 0
        "max"     : round(numbers.max(),  4),
        "count"   : len(numbers),
        "lambda"  : round(1 / mean, 6),         # rate parameter = 1/mean
        # Theoretical skewness of exponential = always 2.0
        # measures how much the right tail dominates
        "skewness": round(float(
            np.mean(((numbers - numbers.mean()) / numbers.std()) ** 3)
        ), 4)
    }

result = generate_exponential(mean=10, count=1000, seed=42)
print({k: v for k, v in result.items() if k != "data"})

{'mean': np.float64(10.1436), 'std': np.float64(10.2709), 'min': np.float64(0.011), 'max': np.float64(74.7067), 'count': 1000, 'lambda': 0.1, 'skewness': 1.9463}


In [ ]:
import numpy as np
import pandas as pd
import math

# ── Redefine function ─────────────────────────────────────────────────────────
def generate_exponential_expert(mean: float, count: int, seed: int = None) -> dict:
    assert mean > 0 and count > 0
    rng     = np.random.default_rng(seed)
    numbers = rng.exponential(scale=mean, size=count)
    return {
        "data"    : numbers,
        "mean"    : round(numbers.mean(), 4),
        "std"     : round(numbers.std(),  4),
        "min"     : round(numbers.min(),  4),
        "max"     : round(numbers.max(),  4),
        "count"   : len(numbers),            # ← lowercase 'c'
        "lambda"  : round(1 / mean, 6),
        "skewness": round(float(
            np.mean(((numbers - numbers.mean()) / numbers.std()) ** 3)
        ), 4)
    }

# ── Generate both sequences ───────────────────────────────────────────────────
seq_10  = generate_exponential_expert(mean=10, count=10,  seed=42)
seq_100 = generate_exponential_expert(mean=10, count=100, seed=42)

# ── Comparison table ──────────────────────────────────────────────────────────
metrics = ["mean", "std", "min", "max", "skewness"]
df_compare = pd.DataFrame({
    "Metric"     : metrics,
    "size=10"    : [seq_10[m]  for m in metrics],
    "size=100"   : [seq_100[m] for m in metrics],
    "Theoretical": [10.0, 10.0, 0.0, "—", 2.0],
})
print(df_compare.to_string(index=False))

# ── Confidence intervals ──────────────────────────────────────────────────────
for name, result in [("size=10", seq_10), ("size=100", seq_100)]:
    # standard error = std / sqrt(n) — note: "count" lowercase here ✅
    se      = result["std"] / math.sqrt(result["count"])   
    ci_low  = result["mean"] - 1.96 * se
    ci_high = result["mean"] + 1.96 * se
    print(f"{name}: mean={result['mean']:.2f} | 95% CI = [{ci_low:.2f}, {ci_high:.2f}]")

  Metric  size=10  size=100 Theoretical
    mean  14.6042    8.9905        10.0
     std  10.3168    8.8499        10.0
     min   0.7929    0.2114         0.0
     max  31.2430   53.0988           —
skewness  -0.0033    1.9928         2.0
size=10: mean=14.60 | 95% CI = [8.21, 21.00]
size=100: mean=8.99 | 95% CI = [7.26, 10.73]


How are the two distributions different?

The mean changes, so the distribution changes as well. 

## Exponential Distribution of Real Data

Suppose that the amount of time one spends in a bank is exponentially distributed with mean as 10 minutes (i.e. λ = 1/10). What is the probability that a customer will spend less than fifteen minutes in the bank? 

Write a code in python to solve this problem

In [33]:
# your code here
from scipy.stats import gaussian_kde

def generate_exponential_expert(mean, count, seed=None):
    assert mean > 0 and count > 0
    rng = np.random.default_rng(seed)
    numbers = rng.exponential(scale=mean, size=count)
    return {
        "data"    : numbers,
        "mean"    : round(numbers.mean(), 4),
        "std"     : round(numbers.std(),  4),
        "count"   : len(numbers),
        "lambda"  : round(1 / mean, 6),
    }

seq_10  = generate_exponential_expert(mean=10, count=10,  seed=42)
seq_100 = generate_exponential_expert(mean=10, count=100, seed=42)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["size=10 — too few samples",
                                    "size=100 — shape emerges"])

lam = 1 / 10   # lambda = 1/mean

for result, col in [(seq_10, 1), (seq_100, 2)]:
    data = result["data"]   # extract numpy array from dict

    # Layer 1: density histogram (100 bins as required)
    fig.add_trace(go.Histogram(
        x=data, nbinsx=100,
        histnorm="probability density",
        opacity=0.55, name=f"Histogram n={result['count']}"
    ), row=1, col=col)

    # Layer 2: KDE — only meaningful with enough data
    # with n=10, KDE is unreliable; with n=100 it starts to work
    if result["count"] >= 10:
        kde   = gaussian_kde(data)
        x_kde = np.linspace(0, data.max(), 300)
        fig.add_trace(go.Scatter(
            x=x_kde, y=kde(x_kde),
            mode="lines", line=dict(width=2),
            name=f"KDE n={result['count']}"
        ), row=1, col=col)

    # Layer 3: theoretical PDF — f(x) = lambda * e^(-lambda * x)
    x_pdf = np.linspace(0, data.max() + 5, 300)
    y_pdf = lam * np.exp(-lam * x_pdf)
    fig.add_trace(go.Scatter(
        x=x_pdf, y=y_pdf,
        mode="lines", line=dict(dash="dash", width=2),
        name="Theoretical PDF"
    ), row=1, col=col)

    # Layer 4: mean line — for exponential, mean = 1/lambda = 10
    fig.add_vline(
        x=result["mean"], line_dash="dot", line_width=2,
        annotation_text=f"mean={result['mean']:.1f}",
        annotation_position="top right",
        row=1, col=col
    )

fig.update_layout(
    title="Exponential Distribution — Expert",
    legend=dict(orientation="h", yanchor="bottom", y=1.05,
                xanchor="center", x=0.5)
)
for col in [1, 2]:
    fig.update_xaxes(title_text="Value",   row=1, col=col)
    fig.update_yaxes(title_text="Density", row=1, col=col)

fig.show()

In [34]:
# your answer here
# Hint: This is same as saying P(x<15)
import numpy as np
import math
from scipy.stats import skew, kurtosis, ks_2samp

np.random.seed(42)
seq_10  = np.random.exponential(scale=10, size=10)
seq_100 = np.random.exponential(scale=10, size=100)

datasets = [("size=10", seq_10), ("size=100", seq_100)]

print("Full Statistical Comparison")
print("=" * 60)
for name, data in datasets:
    se = data.std() / math.sqrt(len(data))
    print(f"\n{name}:")
    print(f"  Mean       : {data.mean():.4f}  (true = 10.0)")
    print(f"  Std Dev    : {data.std():.4f}  (true ≈ 10.0)")

    # Skewness: exponential always has theoretical skewness = 2.0
    # small sample → noisy estimate far from 2.0
    # large sample → estimate converges toward 2.0
    print(f"  Skewness   : {skew(data):.4f}  (theoretical = 2.0)")

    # Excess kurtosis: exponential theoretical = 6.0
    # measures how heavy the right tail is
    print(f"  Kurtosis   : {kurtosis(data):.4f}  (theoretical = 6.0)")
    print(f"  Std Error  : {se:.4f}")
    print(f"  95% CI     : [{data.mean()-1.96*se:.2f}, {data.mean()+1.96*se:.2f}]")

# KS test: do both samples come from the same distribution?
# H0: both sequences ARE drawn from the same distribution
# p > 0.05 → same distribution (only sample size differs)
# p < 0.05 → statistically different (unlikely here)
stat, p = ks_2samp(seq_10, seq_100)
print(f"\nKolmogorov-Smirnov Test (are they from same distribution?):")
print(f"  KS stat = {stat:.4f} | p-value = {p:.4f}")
print(f"  → {'Same distribution (p>0.05)' if p > 0.05 else 'Different (p<0.05)'}")
print("  Note: both come from Exp(mean=10) — only sample size differs")

Full Statistical Comparison

size=10:
  Mean       : 10.2697  (true = 10.0)
  Std Dev    : 8.8139  (true ≈ 10.0)
  Skewness   : 0.9412  (theoretical = 2.0)
  Kurtosis   : 0.0601  (theoretical = 6.0)
  Std Error  : 2.7872
  95% CI     : [4.81, 15.73]

size=100:
  Mean       : 8.8282  (true = 10.0)
  Std Dev    : 8.9967  (true ≈ 10.0)
  Skewness   : 1.5778  (theoretical = 2.0)
  Kurtosis   : 2.3021  (theoretical = 6.0)
  Std Error  : 0.8997
  95% CI     : [7.06, 10.59]

Kolmogorov-Smirnov Test (are they from same distribution?):
  KS stat = 0.2500 | p-value = 0.5581
  → Same distribution (p>0.05)
  Note: both come from Exp(mean=10) — only sample size differs


In [35]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import expon

mean = 10          # average bank wait time in minutes
lam  = 1 / mean    # lambda = rate = 0.1
x    = 15          # threshold in minutes

# ── Core Calculation ──────────────────────────────────────────────────────────
# CDF: P(X < x) = 1 - e^(-lambda * x)
# This is the area under the PDF curve from 0 to x
p_less = expon.cdf(x, loc=0, scale=mean)       # P(X < 15)
p_more = 1 - p_less                            # P(X > 15) — for Question 5.2

print("=" * 55)
print(f"Exponential Distribution — Bank Problem")
print(f"  Mean wait time  : {mean} minutes")
print(f"  Lambda (rate)   : {lam} (events per minute)")
print("=" * 55)
print(f"\nP(X < 15 min)    = 1 - e^(-{lam}×{x})")
print(f"                 = 1 - e^(-{lam*x})")
print(f"                 = 1 - {np.exp(-lam*x):.6f}")
print(f"                 = {p_less:.6f}")
print(f"                 ≈ {p_less*100:.2f}%")

# ── Visualization: shade P(X < 15) under the PDF curve ───────────────────────
x_vals = np.linspace(0, 60, 400)
y_vals = lam * np.exp(-lam * x_vals)           # PDF: f(x) = lambda * e^(-lambda*x)

x_fill = np.linspace(0, x, 200)                # region to shade (0 to 15)
y_fill = lam * np.exp(-lam * x_fill)

fig = go.Figure()

# Full PDF curve
fig.add_trace(go.Scatter(
    x=x_vals, y=y_vals,
    mode="lines", line=dict(width=2),
    name="PDF: f(x) = λe^(-λx)"
))

# Shaded area = P(X < 15)
fig.add_trace(go.Scatter(
    x=np.concatenate([x_fill, x_fill[::-1]]),
    y=np.concatenate([y_fill, np.zeros(len(y_fill))]),
    fill="toself", opacity=0.4,
    name=f"P(X < 15) = {p_less:.4f}"
))

# Vertical threshold line at x=15
fig.add_vline(x=x, line_dash="dash", line_width=2,
              annotation_text="x = 15 min",
              annotation_position="top right")

fig.update_layout(
    title={"text": "Bank Wait Time — P(X < 15 min)<br>"
                   "<span style='font-size:16px;font-weight:normal'>"
                   f"Shaded area = {p_less*100:.2f}% | Exp(mean=10)</span>"},
    legend=dict(orientation="h", yanchor="bottom", y=1.05,
                xanchor="center", x=0.5)
)
fig.update_xaxes(title_text="Time (min)")
fig.update_yaxes(title_text="Density")
fig.show()


Exponential Distribution — Bank Problem
  Mean wait time  : 10 minutes
  Lambda (rate)   : 0.1 (events per minute)

P(X < 15 min)    = 1 - e^(-0.1×15)
                 = 1 - e^(-1.5)
                 = 1 - 0.223130
                 = 0.776870
                 ≈ 77.69%


What is the probability that the customer will spend more than 15 minutes

In [36]:
# your code here
import numpy as np
import plotly.graph_objects as go
from scipy.stats import expon

mean = 10
lam  = 1 / mean
x    = 15

# ── Compute both probabilities ─────────────────────────────────────────────
p_less = expon.cdf(x, loc=0, scale=mean)   # P(X < 15) from Task 5.1
p_more = expon.sf(x,  loc=0, scale=mean)   # P(X > 15) — survival function

print("=" * 55)
print("Complete Bank Wait Time Probability Summary")
print("=" * 55)
print(f"  Distribution  : Exponential(mean={mean} min)")
print(f"  Lambda        : {lam} events/min")
print(f"  Threshold     : {x} minutes")
print("-" * 55)
print(f"  P(X < 15)     = 1 - e^(-{lam*x}) = {p_less:.6f} ({p_less*100:.2f}%)")
print(f"  P(X > 15)     =     e^(-{lam*x}) = {p_more:.6f} ({p_more*100:.2f}%)")
print(f"  P(X < 15) + P(X > 15) = {p_less + p_more:.6f} (must = 1.0)")

# ── Visualization: show BOTH regions under the curve ──────────────────────────
x_vals = np.linspace(0, 60, 400)
y_vals = lam * np.exp(-lam * x_vals)     # full PDF: f(x) = lambda * e^(-lambda*x)

# Region 1: P(X < 15) — left of threshold
x_left = np.linspace(0, x, 200)
y_left = lam * np.exp(-lam * x_left)

# Region 2: P(X > 15) — right of threshold
x_right = np.linspace(x, 60, 200)
y_right = lam * np.exp(-lam * x_right)

fig = go.Figure()

# Full PDF curve
fig.add_trace(go.Scatter(
    x=x_vals, y=y_vals,
    mode="lines", line=dict(width=2),
    name="PDF: f(x) = λe^(-λx)"
))

# Shade P(X < 15) — blue region
fig.add_trace(go.Scatter(
    x=np.concatenate([x_left, x_left[::-1]]),
    y=np.concatenate([y_left, np.zeros(len(y_left))]),
    fill="toself", opacity=0.5,
    name=f"P(X < 15) = {p_less:.4f} ({p_less*100:.1f}%)"
))

# Shade P(X > 15) — red region (answer to this question)
fig.add_trace(go.Scatter(
    x=np.concatenate([x_right, x_right[::-1]]),
    y=np.concatenate([y_right, np.zeros(len(y_right))]),
    fill="toself", opacity=0.5,
    name=f"P(X > 15) = {p_more:.4f} ({p_more*100:.1f}%)"
))

# Threshold line
fig.add_vline(x=x, line_dash="dash", line_width=2,
              annotation_text="x = 15 min",
              annotation_position="top right")

# Mean line — expected wait time
fig.add_vline(x=mean, line_dash="dot", line_width=1,
              annotation_text=f"mean = {mean} min",
              annotation_position="top left")

fig.update_layout(
    title={"text": "Bank Wait Time — Full Probability Split<br>"
                   "<span style='font-size:16px;font-weight:normal'>"
                   f"P(X<15)={p_less*100:.1f}% | P(X>15)={p_more*100:.1f}%</span>"},
    legend=dict(orientation="h", yanchor="bottom", y=1.05,
                xanchor="center", x=0.5)
)
fig.update_xaxes(title_text="Time (min)")
fig.update_yaxes(title_text="Density")
fig.show()



Complete Bank Wait Time Probability Summary
  Distribution  : Exponential(mean=10 min)
  Lambda        : 0.1 events/min
  Threshold     : 15 minutes
-------------------------------------------------------
  P(X < 15)     = 1 - e^(-1.5) = 0.776870 (77.69%)
  P(X > 15)     =     e^(-1.5) = 0.223130 (22.31%)
  P(X < 15) + P(X > 15) = 1.000000 (must = 1.0)
